# DAV LAB 01 | 24I-2605 Shoaib Ali Kori

In [17]:
import pandas as pd
import numpy as np
import sqlite3

df = pd.read_csv('adult.csv')

### TASK 01

In [3]:
print("Initial Info :")
df.info()

print("Initial isnull().sum() count:")
print(df.isnull().sum())


df.replace([' ?', '?'], np.nan, inplace=True)

print("Re-checking isnull().sum() after replacement:")
print(df.isnull().sum())

Initial Info :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       30725 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      30718 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  31978 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
Initial isnull().sum() count:
age                  0
workclass         1836
fn

### TASK 02

In [6]:
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentages
}).sort_values(by='Missing Count', ascending=False)

print("Missing Data Summary : ")
print(missing_summary)

Missing Data Summary : 
                Missing Count  Percentage (%)
occupation               1843            5.66
workclass                1836            5.64
native.country            583            1.79
fnlwgt                      0            0.00
education                   0            0.00
education.num               0            0.00
age                         0            0.00
marital.status              0            0.00
relationship                0            0.00
sex                         0            0.00
race                        0            0.00
capital.gain                0            0.00
capital.loss                0            0.00
hours.per.week              0            0.00
income                      0            0.00


### Task 03

In [7]:
df['workclass'].fillna('Unknown', inplace=True)
df['occupation'].fillna('Unknown', inplace=True)
df['native.country'].fillna(df['native.country'].mode()[0], inplace=True)

# Verify no nulls remain
print("Remaining null values after imputation:", df.isnull().sum().sum())

# Justifications:
# 1. workclass & occupation: Imputed with 'Unknown' because dropping missing entries removes 
#    over 2,000 valuable records (~7% of the dataset) and preserves the missingness relationship.
# 2. native.country: Imputed with mode ('United-States') because over 89% of respondents belong 
#    to this category, making mode imputation statistically reasonable.
# 3. Numeric Columns: No numeric missing values were present in this dataset.

Remaining null values after imputation: 0


### Task 04

In [8]:
exact_duplicates = df.duplicated().sum()
print(f"Exact Duplicate Rows Count: {exact_duplicates}")

income_col = 'income' if 'income' in df.columns else df.columns[-1]
feature_cols = [col for col in df.columns if col != income_col]
feature_duplicates = df.duplicated(subset=feature_cols).sum()
print(f"Duplicate Rows Ignoring Label Column ('{income_col}'): {feature_duplicates}")
df.drop_duplicates(inplace=True)
print(f"Dataset shape after dropping exact duplicates: {df.shape}")

# Decision Justification:
# Exact duplicate rows represent repeated data entries and should be dropped. 
# Rows with identical demographic features but differing income labels are NOT dropped, 
# because two individuals can share identical demographic traits yet earn different incomes.


Exact Duplicate Rows Count: 24
Duplicate Rows Ignoring Label Column ('income'): 25
Dataset shape after dropping exact duplicates: (32537, 15)


### Task 05

In [9]:
edu_col = 'education'
marital_col = 'marital.status' if 'marital.status' in df.columns else 'marital-status'
country_col = 'native.country' if 'native.country' in df.columns else 'native-country'

print(" Unique Values Before Stripping ")
print("Education:", df[edu_col].unique()[:5])
print("Marital Status:", df[marital_col].unique()[:5])
print("Native Country:", df[country_col].unique()[:5])

categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

print("\nUnique Values After Stripping ")
print("Education:", df[edu_col].unique()[:5])
print("Marital Status:", df[marital_col].unique()[:5])
print("Native Country:", df[country_col].unique()[:5])

 Unique Values Before Stripping 
Education: ['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate']
Marital Status: ['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse']
Native Country: ['United-States' 'Mexico' 'Greece' 'Vietnam' 'China']

Unique Values After Stripping 
Education: ['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate']
Marital Status: ['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse']
Native Country: ['United-States' 'Mexico' 'Greece' 'Vietnam' 'China']


### Task 06

In [13]:
print(" Numeric Descriptive Statistics ")
print(df.describe())

print("\n Categorical Summary Statistics ")
print(df.describe(include='object'))
hpw_col = 'hours.per.week' if 'hours.per.week' in df.columns else 'hours-per-week'

age_mean = df['age'].mean()
age_median = df['age'].median()
hpw_mean = df[hpw_col].mean()
hpw_median = df[hpw_col].median()

print(f"\nAge -> Mean: {age_mean:.2f}, Median: {age_median}")
print(f"Hours-Per-Week -> Mean: {hpw_mean:.2f}, Median: {hpw_median}")


 Numeric Descriptive Statistics 
                age        fnlwgt  education.num  capital.gain  capital.loss  \
count  32537.000000  3.253700e+04   32537.000000  32537.000000  32537.000000   
mean      38.585549  1.897808e+05      10.081815   1078.443741     87.368227   
std       13.637984  1.055565e+05       2.571633   7387.957424    403.101833   
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000   
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000   
50%       37.000000  1.783560e+05      10.000000      0.000000      0.000000   
75%       48.000000  2.369930e+05      12.000000      0.000000      0.000000   
max       90.000000  1.484705e+06      16.000000  99999.000000   4356.000000   

       hours.per.week  
count    32537.000000  
mean        40.440329  
std         12.346889  
min          1.000000  
25%         40.000000  
50%         40.000000  
75%         45.000000  
max         99.000000  

 Categorical Summary Statisti

### Task 07

In [14]:
# (a)
most_common_occ = df['occupation'].value_counts().idxmax()
print(f"Most Common Occupation: {most_common_occ}")

# (b) 
print("\nPercentage Distribution of Sex (%):")
print((df['sex'].value_counts(normalize=True) * 100).round(2))

# (c)
print(f"\nPercentage Distribution of Income Target ('{income_col}') (%):")
income_dist = (df[income_col].value_counts(normalize=True) * 100).round(2)
print(income_dist)

# Comment on balance:
# The income classes are heavily imbalanced. 

Most Common Occupation: Prof-specialty

Percentage Distribution of Sex (%):
sex
Male      66.92
Female    33.08
Name: proportion, dtype: float64

Percentage Distribution of Income Target ('income') (%):
income
<=50K    75.91
>50K     24.09
Name: proportion, dtype: float64


### Task 08

In [15]:
edu_num_col = 'education.num' if 'education.num' in df.columns else 'education-num'

mapping_check = df.groupby('education')[edu_num_col].unique()
print("Education Mapping Verification")
print(mapping_check)

# Assert strict 1-to-1 consistency
is_consistent = all(len(num_array) == 1 for num_array in mapping_check)
print(f"\nAre 'education' and '{edu_num_col}' strictly consistent with 1-to-1 mapping? {is_consistent}")

Education Mapping Verification
education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education.num, dtype: object

Are 'education' and 'education.num' strictly consistent with 1-to-1 mapping? True


### Task 09

In [16]:
raw_df = pd.read_csv('adult.csv')
conn = sqlite3.connect(':memory:')
raw_df.to_sql('adult_income', conn, index=False, if_exists='replace')

sql_query = "SELECT * FROM adult_income WHERE age > 30;"
sql_extracted_df = pd.read_sql_query(sql_query, conn)
pandas_filtered_df = raw_df[raw_df['age'] > 30]

print(f"SQL Extracted DataFrame Shape:    {sql_extracted_df.shape}")
print(f"Pandas Filtered DataFrame Shape: {pandas_filtered_df.shape}")

assert sql_extracted_df.shape == pandas_filtered_df.shape, "Shapes do not match!"
print("\nExtraction Verified: SQL query matches Pandas filter results perfectly.")

conn.close()

SQL Extracted DataFrame Shape:    (21989, 15)
Pandas Filtered DataFrame Shape: (21989, 15)

Extraction Verified: SQL query matches Pandas filter results perfectly.


### Task 10

# Summary Report

## 1. Dataset Overview
* **Dimensions:** 32,561 initial rows and 15 columns.
* **Representation:** Demographic, educational, and employment attributes from census data aimed at predicting whether an individual earns more than $50K annually.

## 2. Data Quality Issues Found
* **Hidden Missing Values:** 4,262 entries represented as string `' ?'` or `'?'` across `occupation` (1,843), `workclass` (1,836), and `native-country` (583).
* **Exact Duplicate Rows:** 24 exact duplicate records present across all attributes.
* **Categorical Whitespace:** Leading/trailing spaces present across object columns preventing proper group aggregation.

## 3. Cleaning Decisions Made
* **Missing Data:** Imputed `workclass` and `occupation` with `'Unknown'` to preserve row count; imputed `native-country` with mode (`'United-States'`).
* **Duplicates:** Removed 24 exact duplicate rows using `.drop_duplicates()`.
* **String Standardization:** Applied `.str.strip()` across all categorical columns to clean whitespace formatting.

## 4. Key Observations
* **Class Imbalance:** Income target is imbalanced with 75.91% earning `<=50K` and 24.09% earning `>50K`.
* **Occupations & Sex:** `Prof-specialty` is the most common occupation; workforce demographics show a ~66.92% Male to ~33.08% Female distribution.
* **Feature Consistency:** Cross-checking confirmed that `education` text strictly maps 1-to-1 to numeric `education-num` levels.